# 2cmab2 运行结果复盘 Notebook

这个 notebook 用来分析 `2cmab2/outputs/...` 下面某一次训练输出。

使用方式：

1. 先把下面配置单元里的 `RUN_DIR` 改成你要分析的输出目录。
2. 顺序执行全部单元。
3. 如果希望把 notebook 里的表格同时导出成 markdown / csv，就把 `EXPORT_ANALYSIS = True`。

这个 notebook 会重点输出：

- selector / baseline 指标
- top1 / top2 / top3 命中率同表展示
- test 集各方法自身的 mean cost，以及 selector 相比它们更好还是更差
- 选臂分布与选择集中度
- 每个 arm 被选中后的真实效果
- greedy 和 ucb 的差异
- oracle 最优 arm 分布与覆盖情况

说明：

- 所有表格默认只展示按问题拆分后的结果，例如 `tsp`、`cvrp`


In [3]:
from pathlib import Path
import sys
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd()
MODULE_DIR = REPO_ROOT / "2cmab2"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import output_analysis_utils as au

# 改这里：要分析的 run 输出目录
RUN_DIR = REPO_ROOT / "outputs/2026-03-23-23-07-19_neural_linucb"

# 如果你希望 notebook 同时把表格导出到 run_dir/notebook_analysis/ 下面，就设成 True
EXPORT_ANALYSIS = True
EXPORT_DIR = RUN_DIR / "notebook_analysis"


In [4]:
analysis = au.build_analysis_bundle(RUN_DIR)
display(Markdown("# 运行结果总览"))
display(Markdown("\n".join(analysis["overview_lines"])))
display(Markdown("## 阶段耗时"))
display(Markdown(au.rows_to_markdown_table(
    analysis["runtime"]["runtime_rows"],
    columns=["stage", "seconds", "readable"],
)))


/home/viayu/miniconda3/envs/easynco/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 运行结果总览

- run 目录: `/mnt/d/Study/3/neural-solver-selection/2cmab2/outputs/2026-03-23-23-07-19_neural_linucb`
- method: `neural_linucb`
- reward_mode: `linear_zero_one`
- train/val/test: `14000 / 3000 / 3000`
- alpha: `1.0`, hidden_dim: `64`, train_every: `50`, representation_steps: `10`
- representation_buffer_size: `5000`, linear_head_buffer_size: `-1`
- freeze_encoder: `False`
- checkpoint 文件数: `0`
- 训练耗时: `5h 1m 44s`

## 阶段耗时

| stage | seconds | readable |
| --- | --- | --- |
| dataset_build | 49 | 49s |
| training | 18104 | 5h 1m 44s |
| selector_eval | 67 | 1m 7s |
| baseline_eval | 1 | 1s |
| write_results | 0 | 0s |

In [5]:
display(Markdown("## Selector 指标"))
display(Markdown(au.rows_to_markdown_table(
    analysis["selector_rows"],
    columns=["source", "split", "scope", "count", "mean_reward", "top1_accuracy", "mean_regret", "mean_cost"],
)))

display(Markdown("## Baseline 指标"))
display(Markdown(au.rows_to_markdown_table(
    analysis["baseline_rows"],
    columns=["source", "split", "scope", "count", "mean_reward", "top1_accuracy", "mean_regret", "mean_cost"],
)))


## Selector 指标

| source | split | scope | count | mean_reward | top1_accuracy | mean_regret | mean_cost |
| --- | --- | --- | --- | --- | --- | --- | --- |
| selector | val_greedy | cvrp | 1500 | 0.832000 | 0.362667 | 0.084903 | 15.856593 |
| selector | val_greedy | tsp | 1500 | 0.852758 | 0.262000 | 0.010122 | 7.775747 |
| selector | test_greedy | cvrp | 1500 | 0.831429 | 0.358667 | 0.087679 | 15.850527 |
| selector | test_greedy | tsp | 1500 | 0.850515 | 0.248000 | 0.009892 | 7.771475 |
| selector | val_ucb | cvrp | 1500 | 0.832000 | 0.360667 | 0.084822 | 15.856513 |
| selector | val_ucb | tsp | 1500 | 0.851242 | 0.268667 | 0.010665 | 7.776290 |
| selector | test_ucb | cvrp | 1500 | 0.831143 | 0.352667 | 0.091307 | 15.854155 |
| selector | test_ucb | tsp | 1500 | 0.853485 | 0.262667 | 0.009755 | 7.771337 |

## Baseline 指标

| source | split | scope | count | mean_reward | top1_accuracy | mean_regret | mean_cost |
| --- | --- | --- | --- | --- | --- | --- | --- |
| random | test_greedy | cvrp | 1500 | 0.492952 | 0.124000 | 2.285205 | 18.048054 |
| random | test_greedy | tsp | 1500 | 0.504788 | 0.088000 | 3.696003 | 11.457586 |
| single_best_global | test_greedy | cvrp | 1500 | 0.831619 | 0.411333 | 0.090761 | 15.853609 |
| single_best_global | test_greedy | tsp | 1500 | 0.846273 | 0.139333 | 0.010797 | 7.772379 |
| single_best_per_problem | test_greedy | cvrp | 1500 | 0.831619 | 0.411333 | 0.090761 | 15.853609 |
| single_best_per_problem | test_greedy | tsp | 1500 | 0.863727 | 0.300667 | 0.009191 | 7.770773 |
| oracle | test_greedy | cvrp | 1500 | 1 | 1 | 0 | 15.762848 |
| oracle | test_greedy | tsp | 1500 | 0.981485 | 1 | 0 | 7.761582 |

In [6]:
display(Markdown(
    "## Top-k 命中分析\n"
    "- 这里的 `top1/top2/top3` 是根据 trace 中的 `selected_reward` 反推出选中 arm 的 tie-aware 平均 rank 后计算的。\n"
    "- 因此它适合回答“模型选中的方法有没有落在真实前 1 / 前 2 / 前 3 名”。\n"
    "- 它和 `summary.json` 里的 `top1_accuracy` 定义不同，后者是 `selected_arm == best_arm`。"
))

for trace_name, rows in analysis["topk_by_trace"].items():
    display(Markdown(f"### {trace_name}"))
    display(Markdown(au.rows_to_markdown_table(
        rows,
        columns=["scope", "count", "mean_selected_rank", "top1_hit_rate", "top2_hit_rate", "top3_hit_rate"],
    )))


## Top-k 命中分析
- 这里的 `top1/top2/top3` 是根据 trace 中的 `selected_reward` 反推出选中 arm 的 tie-aware 平均 rank 后计算的。
- 因此它适合回答“模型选中的方法有没有落在真实前 1 / 前 2 / 前 3 名”。
- 它和 `summary.json` 里的 `top1_accuracy` 定义不同，后者是 `selected_arm == best_arm`。

### selector_train_ucb.jsonl

| scope | count | mean_selected_rank | top1_hit_rate | top2_hit_rate | top3_hit_rate |
| --- | --- | --- | --- | --- | --- |
| cvrp | 7000 | 2.192000 | 0.348857 | 0.619143 | 0.852571 |
| tsp | 7000 | 2.637286 | 0.189143 | 0.504571 | 0.719571 |

### selector_val_greedy.jsonl

| scope | count | mean_selected_rank | top1_hit_rate | top2_hit_rate | top3_hit_rate |
| --- | --- | --- | --- | --- | --- |
| cvrp | 1500 | 2.176000 | 0.362667 | 0.614667 | 0.854000 |
| tsp | 1500 | 2.619667 | 0.191333 | 0.500667 | 0.722000 |

...

| scope | count | mean_selected_rank | top1_hit_rate | top2_hit_rate | top3_hit_rate |
| --- | --- | --- | --- | --- | --- |
| cvrp | 1500 | 2.176000 | 0.360667 | 0.612667 | 0.855333 |
| tsp | 1500 | 2.636333 | 0.188000 | 0.500000 | 0.714667 |

### selector_test_greedy.jsonl

| scope | count | mean_selected_rank | top1_hit_rate | top2_hit_rate | top3_hit_rate |
| --- | --- | --- | --- | --- | --- |
| cvrp | 1500 | 2.180000 | 0.358667 | 0.611333 | 0.858667 |
| tsp | 1500 | 2.644333 | 0.184000 | 0.487333 | 0.712667 |

### selector_test_ucb.jsonl

| scope | count | mean_selected_rank | top1_hit_rate | top2_hit_rate | top3_hit_rate |
| --- | --- | --- | --- | --- | --- |
| cvrp | 1500 | 2.182000 | 0.352667 | 0.618667 | 0.858667 |
| tsp | 1500 | 2.611667 | 0.195333 | 0.500667 | 0.716000 |

In [7]:
for trace_name, rows in analysis["selection_summary_by_trace"].items():
    display(Markdown(f"## {trace_name} 选择集中度"))
    display(Markdown(au.rows_to_markdown_table(
        rows,
        columns=["scope", "count", "unique_selected_arms", "entropy", "effective_arms"],
    )))

for trace_name, rows in analysis["selection_distribution_by_trace"].items():
    display(Markdown(f"## {trace_name} 各 arm 被选比例"))
    display(Markdown(au.rows_to_markdown_table(
        rows,
        columns=["scope", "arm", "count", "fraction"],
    )))


## selector_train_ucb.jsonl 选择集中度

| scope | count | unique_selected_arms | entropy | effective_arms |
| --- | --- | --- | --- | --- |
| tsp | 7000 | 11 | 1.192955 | 3.296809 |
| cvrp | 7000 | 8 | 1.014897 | 2.759079 |

## selector_val_greedy.jsonl 选择集中度

| scope | count | unique_selected_arms | entropy | effective_arms |
| --- | --- | --- | --- | --- |
| cvrp | 1500 | 4 | 1.027027 | 2.792751 |
| tsp | 1500 | 4 | 1.170366 | 3.223173 |

## selector_val_ucb.jsonl 选择集中度

...

| scope | arm | count | fraction |
| --- | --- | --- | --- |
| cvrp | lehd | 719 | 0.479333 |
| cvrp | icam | 536 | 0.357333 |
| cvrp | elg | 235 | 0.156667 |
| cvrp | omni | 10 | 0.006667 |
| tsp | icam | 686 | 0.457333 |
| tsp | pointerformer | 506 | 0.337333 |
| tsp | lehd | 251 | 0.167333 |
| tsp | elg | 55 | 0.036667 |
| tsp | omni | 2 | 0.001333 |

## selector_test_greedy.jsonl 各 arm 被选比例

| scope | arm | count | fraction |
| --- | --- | --- | --- |
| cvrp | lehd | 792 | 0.528000 |
| cvrp | icam | 456 | 0.304000 |
| cvrp | elg | 252 | 0.168000 |
| tsp | icam | 617 | 0.411333 |
| tsp | pointerformer | 553 | 0.368667 |
| tsp | lehd | 281 | 0.187333 |
| tsp | elg | 49 | 0.032667 |

## selector_test_ucb.jsonl 各 arm 被选比例

| scope | arm | count | fraction |
| --- | --- | --- | --- |
| cvrp | lehd | 729 | 0.486000 |
| cvrp | icam | 572 | 0.381333 |
| cvrp | elg | 188 | 0.125333 |
| cvrp | omni | 10 | 0.006667 |
| cvrp | dact | 1 | 0.000667 |
| tsp | icam | 686 | 0.457333 |
| tsp | pointerformer | 497 | 0.331333 |
| tsp | lehd | 258 | 0.172000 |
| tsp | elg | 58 | 0.038667 |
| tsp | omni | 1 | 0.000667 |

In [8]:
for trace_name, rows in analysis["selected_arm_quality_by_trace"].items():
    display(Markdown(f"## {trace_name} 每个 arm 被选中后的真实效果"))
    display(Markdown(au.rows_to_markdown_table(
        rows,
        columns=["scope", "arm", "count", "mean_reward", "mean_regret", "top1_rate", "mean_cost"],
    )))

if analysis.get("method_mean_cost_error"):
    display(Markdown("## test 集各方法 mean cost 与 selector 对比"))
    display(Markdown(f"- 该部分生成失败：`{analysis['method_mean_cost_error']}`"))
else:
    for trace_name, rows in analysis["selector_vs_methods_summary_by_trace"].items():
        display(Markdown(f"## {trace_name} selector 与各方法 mean cost 对比汇总"))
        display(Markdown(au.rows_to_markdown_table(
            rows,
            columns=[
                "scope",
                "selector_mean_cost",
                "n_methods",
                "methods_better_than_selector",
                "methods_worse_than_selector",
                "methods_equal_to_selector",
                "selector_rank_among_methods",
            ],
        )))

    for trace_name, rows in analysis["test_method_mean_cost_by_trace"].items():
        display(Markdown(f"## {trace_name} 各方法在 test 集上的 mean cost"))
        display(Markdown(au.rows_to_markdown_table(
            rows,
            columns=["scope", "arm", "count", "mean_cost", "selector_mean_cost", "delta_vs_selector", "selector_better"],
        )))

display(Markdown("## oracle 最优 arm 分布"))
display(Markdown(au.rows_to_markdown_table(
    analysis["best_arm_rows"],
    columns=["scope", "arm", "oracle_best_count", "oracle_best_fraction"],
)))

display(Markdown("## oracle 覆盖度"))
display(Markdown(au.rows_to_markdown_table(
    analysis["oracle_coverage_rows"],
    columns=["arm", "oracle_best_count", "selected_count", "selected_minus_oracle", "coverage_over_oracle"],
)))


## selector_train_ucb.jsonl 每个 arm 被选中后的真实效果

| scope | arm | count | mean_reward | mean_regret | top1_rate | mean_cost |
| --- | --- | --- | --- | --- | --- | --- |
| cvrp | lehd | 3586 | 0.836706 | 0.087967 | 0.420524 | 15.858771 |
| cvrp | icam | 2388 | 0.827889 | 0.089144 | 0.286013 | 15.897115 |
| cvrp | elg | 994 | 0.817620 | 0.087760 | 0.251509 | 15.907691 |
| cvrp | omni | 25 | 0.651429 | 0.216963 | 0.040000 | 15.919853 |
| cvrp | dact | 4 | 0.214286 | 5.011737 | 0 | 20.282692 |
| cvrp | invit | 1 | 0.428571 | 0.970682 | 0 | 17.573004 |
| cvrp | lih | 1 | 0.285714 | 3.353895 | 0 | 19.415602 |
| cvrp | udc | 1 | 0 | 6.183142 | 0 | 21.233707 |
| tsp | icam | 3036 | 0.863352 | 0.009434 | 0.311924 | 7.786669 |
| tsp | pointerformer | 2305 | 0.856084 | 0.010220 | 0.142733 | 7.768408 |
| tsp | lehd | 1388 | 0.840451 | 0.010234 | 0.352305 | 7.762402 |
| tsp | elg | 245 | 0.769388 | 0.016984 | 0.097959 | 7.793648 |
| tsp | dact | 9 | 0.191919 | 1.789357 | 0 | 9.575404 |
| tsp | omni | 8 | 0.534091 | 0.127213 | 0 | 7.880438 |
| tsp | udc | 4 | 0.113636 | 2.318262 | 0 | 10.036646 |
| tsp | invit | 2 | 0.454545 | 0.173226 | 0 | 8.100258 |
| tsp | difusco | 1 | 0.636364 | 0.062112 | 0 | 7.700190 |
| tsp | glop | 1 | 0.363636 | 1.082900 | 0 | 8.386805 |
| tsp | t2t | 1 | 0.272727 | 0.897561 | 0 | 8.299824 |

## selector_val_greedy.jsonl 每个 arm 被选中后的真实效果

| scope | arm | count | mean_reward | mean_regret | top1_rate | mean_cost |
| --- | --- | --- | --- | --- | --- | --- |
| cvrp | lehd | 774 | 0.838686 | 0.086097 | 0.443152 | 15.820378 |
| cvrp | icam | 418 | 0.827409 | 0.081866 | 0.303828 | 15.886076 |
| cvrp | elg | 307 | 0.820847 | 0.086304 | 0.237785 | 15.896640 |
| cvrp | omni | 1 | 1 | 0 | 1 | 19.269119 |
| tsp | icam | 609 | 0.863562 | 0.008666 | 0.305419 | 7.778001 |
| tsp | pointerformer | 549 | 0.853453 | 0.011160 | 0.156648 | 7.771430 |
| tsp | lehd | 288 | 0.841067 | 0.009988 | 0.385417 | 7.776139 |
| tsp | elg | 54 | 0.786195 | 0.016703 | 0.185185 | 7.792126 |

## selector_val_ucb.jsonl 每个 arm 被选中后的真实效果

...

| scope | arm | count | mean_cost | selector_mean_cost | delta_vs_selector | selector_better |
| --- | --- | --- | --- | --- | --- | --- |
| cvrp | icam | 1500 | 15.849489 | 15.854155 | -0.004666 | false |
| cvrp | lehd | 1500 | 15.853609 | 15.854155 | -0.000546 | false |
| cvrp | elg | 1500 | 15.853644 | 15.854155 | -0.000512 | false |
| cvrp | omni | 1500 | 15.963844 | 15.854155 | 0.109689 | true |
| cvrp | invit | 1500 | 16.353747 | 15.854155 | 0.499592 | true |
| cvrp | lih | 1500 | 20.541339 | 15.854155 | 4.687184 | true |
| cvrp | dact | 1500 | 20.848404 | 15.854155 | 4.994249 | true |
| cvrp | udc | 1500 | 22.648666 | 15.854155 | 6.794510 | true |
| tsp | lehd | 1500 | 7.769759 | 7.771337 | -0.001578 | false |
| tsp | icam | 1500 | 7.770773 | 7.771337 | -0.000565 | false |
| tsp | pointerformer | 1500 | 7.772379 | 7.771337 | 0.001042 | true |
| tsp | elg | 1500 | 7.778870 | 7.771337 | 0.007533 | true |
| tsp | omni | 1500 | 7.858207 | 7.771337 | 0.086870 | true |
| tsp | difusco | 1500 | 7.890429 | 7.771337 | 0.119091 | true |
| tsp | invit | 1500 | 7.977307 | 7.771337 | 0.205969 | true |
| tsp | glop | 1500 | 8.514799 | 7.771337 | 0.743461 | true |
| tsp | t2t | 1500 | 9.169899 | 7.771337 | 1.398562 | true |
| tsp | dact | 1500 | 9.682358 | 7.771337 | 1.911021 | true |
| tsp | udc | 1500 | 10.180803 | 7.771337 | 2.409465 | true |
| tsp | lih | 1500 | 49.078221 | 7.771337 | 41.306883 | true |

## oracle 最优 arm 分布

| scope | arm | oracle_best_count | oracle_best_fraction |
| --- | --- | --- | --- |
| cvrp | lehd | 617 | 0.411333 |
| cvrp | icam | 440 | 0.293333 |
| cvrp | elg | 381 | 0.254000 |
| cvrp | omni | 62 | 0.041333 |
| tsp | lehd | 553 | 0.368667 |
| tsp | icam | 451 | 0.300667 |
| tsp | pointerformer | 209 | 0.139333 |
| tsp | elg | 201 | 0.134000 |
| tsp | difusco | 80 | 0.053333 |
| tsp | omni | 6 | 0.004000 |

## oracle 覆盖度

| arm | oracle_best_count | selected_count | selected_minus_oracle | coverage_over_oracle |
| --- | --- | --- | --- | --- |
| lehd | 1170 | 1073 | -97 | 0.917094 |
| icam | 891 | 1073 | 182 | 1.204265 |
| elg | 582 | 301 | -281 | 0.517182 |
| pointerformer | 209 | 553 | 344 | 2.645933 |
| difusco | 80 | 0 | -80 | 0 |
| omni | 68 | 0 | -68 | 0 |

In [9]:
display(Markdown("## greedy 与 ucb 的差异"))
display(Markdown(au.rows_to_markdown_table(
    analysis["greedy_vs_ucb_summary"],
    columns=["scope", "total", "same_selection", "different_selection", "different_ratio", "ucb_better_when_diff", "ucb_worse_when_diff", "equal_when_diff"],
)))

display(Markdown("### 具体样例"))
display(Markdown(au.rows_to_markdown_table(
    analysis["greedy_vs_ucb_examples"],
    columns=["uid", "problem", "greedy_arm", "greedy_score", "greedy_regret", "ucb_arm", "ucb_score", "ucb_regret", "best_arm", "best_cost"],
)))


## greedy 与 ucb 的差异

| scope | total | same_selection | different_selection | different_ratio | ucb_better_when_diff | ucb_worse_when_diff | equal_when_diff |
| --- | --- | --- | --- | --- | --- | --- | --- |
| cvrp | 1500 | 1247 | 253 | 0.168667 | 125 | 128 | 0 |
| tsp | 1500 | 1153 | 347 | 0.231333 | 170 | 141 | 36 |

### 具体样例

| uid | problem | greedy_arm | greedy_score | greedy_regret | ucb_arm | ucb_score | ucb_regret | best_arm | best_cost |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| cvrp:1010 | cvrp | lehd | 0.835365 | 0 | icam | 0.858946 | 0.288399 | lehd | 18.030581 |
| cvrp:102 | cvrp | elg | 0.845986 | 0.020197 | lehd | 0.913997 | 0.074341 | icam | 15.366686 |
| cvrp:1027 | cvrp | icam | 0.829254 | 0 | elg | 0.888826 | 0.059476 | icam | 13.003302 |
| cvrp:106 | cvrp | icam | 0.830977 | 0.057542 | lehd | 0.854237 | 0.051605 | omni | 13.589450 |
| cvrp:1079 | cvrp | elg | 0.780106 | 0.084965 | icam | 0.801819 | 0 | icam | 19.456673 |
| cvrp:1139 | cvrp | elg | 0.830138 | 0.042837 | lehd | 0.856113 | 0.085478 | icam | 18.019506 |
| cvrp:1243 | cvrp | lehd | 0.896666 | 0 | icam | 0.956898 | 0.027893 | lehd | 17.670795 |
| cvrp:1244 | cvrp | lehd | 0.904365 | 0 | icam | 0.952459 | 0.092880 | lehd | 15.188339 |
| cvrp:1288 | cvrp | lehd | 0.878678 | 0.204666 | icam | 0.931765 | 0.093544 | elg | 20.455551 |
| cvrp:1298 | cvrp | icam | 0.905134 | 0.187460 | elg | 0.977687 | 0 | elg | 18.091629 |
| cvrp:1320 | cvrp | elg | 0.863966 | 0 | icam | 0.888720 | 0.120375 | elg | 16.241957 |
| cvrp:1402 | cvrp | elg | 0.786149 | 0.265333 | lehd | 0.852896 | 0 | lehd | 16.202568 |
| cvrp:1493 | cvrp | elg | 0.704861 | 0.074207 | lehd | 0.806456 | 0.026867 | icam | 18.190746 |
| cvrp:1505 | cvrp | icam | 0.668678 | 0.006146 | omni | 0.736232 | 0 | omni | 13.581478 |
| cvrp:1530 | cvrp | lehd | 0.958406 | 0.106879 | icam | 1.001669 | 0 | icam | 14.160683 |
| cvrp:159 | cvrp | icam | 0.900736 | 0 | elg | 0.956971 | 0.008718 | icam | 16.615356 |
| cvrp:1611 | cvrp | elg | 0.831545 | 0 | icam | 0.854204 | 0.034554 | elg | 16.317635 |
| cvrp:1623 | cvrp | lehd | 0.826874 | 0 | icam | 0.857810 | 0.269540 | lehd | 15.257771 |
| cvrp:1683 | cvrp | lehd | 0.878823 | 0.130753 | icam | 0.937782 | 0.044527 | elg | 12.407223 |
| cvrp:1736 | cvrp | lehd | 0.798334 | 0.705536 | icam | 0.837960 | 0 | icam | 18.275806 |

In [10]:
if EXPORT_ANALYSIS:
    export_paths = au.export_analysis_bundle(analysis, EXPORT_DIR)
    display(Markdown("## 导出结果"))
    display(Markdown("\n".join([f"- `{key}`: `{value}`" for key, value in export_paths.items()])))
else:
    display(Markdown("## 导出结果\n当前未开启导出。把 `EXPORT_ANALYSIS` 改成 `True` 即可。"))


## 导出结果

- `output_dir`: `/mnt/d/Study/3/neural-solver-selection/2cmab2/outputs/2026-03-23-23-07-19_neural_linucb/notebook_analysis`
- `report_md`: `/mnt/d/Study/3/neural-solver-selection/2cmab2/outputs/2026-03-23-23-07-19_neural_linucb/notebook_analysis/report.md`
- `overview_json`: `/mnt/d/Study/3/neural-solver-selection/2cmab2/outputs/2026-03-23-23-07-19_neural_linucb/notebook_analysis/overview.json`